In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import joblib

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "secom_telemetry_modified.csv"
ARTIFACTS_PATH = ROOT / "artifacts"

torch.manual_seed(42)
np.random.seed(42)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

In [2]:
print("SECTION 1: LOAD DATA")

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")
df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

if "label" not in df.columns:
    raise ValueError("Dataset must contain a label column.")

if not set(df["label"].dropna().unique()).issubset({-1, 1}):
    raise ValueError("Expected labels -1 for normal and 1 for defects.")

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

n = len(df)
a = int(n * 0.70)
b = int(n * 0.80)

train_df = df.iloc[:a].copy()
val_df = df.iloc[a:b].copy()
test_df = df.iloc[b:].copy()

print("Total records:", n)
print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))
print("Training defect prevalence:", (train_df["label"] == 1).mean())
print("Validation defect prevalence:", (val_df["label"] == 1).mean())
print("Test defect prevalence:", (test_df["label"] == 1).mean())

SECTION 1: LOAD DATA
Total records: 1567
Training records: 1096
Validation records: 157
Test records: 314
Training defect prevalence: 0.07116788321167883
Validation defect prevalence: 0.05732484076433121
Test defect prevalence: 0.054140127388535034


In [3]:
print("\nSECTION 2: HANDLE LEGACY PS-L SENSORS")


def get_features(part):
    X = part[sensor_cols].copy()

    legacy = X["sensor_0"].eq(0.0).fillna(False)
    sensor1 = X["sensor_1"].copy()

    X["is_legacy_ps_l"] = legacy.astype(np.float32)

    X["legacy_sensor_1_code_100"] = (
        legacy & sensor1.eq(100.0)
    ).astype(np.float32)

    X["legacy_sensor_1_code_200"] = (
        legacy & sensor1.eq(200.0)
    ).astype(np.float32)

    X["legacy_sensor_1_code_300"] = (
        legacy & sensor1.eq(300.0)
    ).astype(np.float32)

    X.loc[legacy, "sensor_0"] = np.nan
    X["sensor_1"] = sensor1.mask(legacy, np.nan)

    y = (part["label"] == 1).astype(np.int64).to_numpy()

    return X, y


X_train_raw, y_train = get_features(train_df)
X_val_raw, y_val = get_features(val_df)
X_test_raw, y_test = get_features(test_df)

flag_cols = [
    "is_legacy_ps_l",
    "legacy_sensor_1_code_100",
    "legacy_sensor_1_code_200",
    "legacy_sensor_1_code_300"
]

numeric_cols = [c for c in sensor_cols if c != "sensor_1"]

usable_cols = [
    c for c in numeric_cols
    if X_train_raw[c].notna().any()
    and X_train_raw[c].nunique(dropna=True) > 1
]

if not usable_cols:
    raise ValueError("No usable continuous sensor features were found.")

print("Continuous features:", len(usable_cols))
print("Categorical indicator features:", len(flag_cols))
print("sensor_1 included as a continuous feature:", "sensor_1" in usable_cols)

assert "sensor_1" not in usable_cols


SECTION 2: HANDLE LEGACY PS-L SENSORS
Continuous features: 467
Categorical indicator features: 4
sensor_1 included as a continuous feature: False


In [4]:
print("\nSECTION 3: PREPROCESS DATA")

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

train_imputed = imputer.fit_transform(X_train_raw[usable_cols])
val_imputed = imputer.transform(X_val_raw[usable_cols])
test_imputed = imputer.transform(X_test_raw[usable_cols])

train_scaled = scaler.fit_transform(train_imputed)
val_scaled = scaler.transform(val_imputed)
test_scaled = scaler.transform(test_imputed)


def combine_features(scaled, raw):
    flags = raw[flag_cols].to_numpy(dtype=np.float32)
    return np.column_stack([scaled, flags]).astype(np.float32)


X_train = combine_features(train_scaled, X_train_raw)
X_val = combine_features(val_scaled, X_val_raw)
X_test = combine_features(test_scaled, X_test_raw)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

print("All features are finite.")
print("Imputer and scaler fitted on training data only.")


SECTION 3: PREPROCESS DATA
Training shape: (1096, 471)
Validation shape: (157, 471)
Test shape: (314, 471)
All features are finite.
Imputer and scaler fitted on training data only.


In [5]:
print("\nSECTION 4: BUILD MODEL")


class WaferAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

        self.classifier = nn.Linear(latent_dim, 1)

    def forward(self, x):
        if x.ndim != 2:
            raise ValueError(
                f"Expected a 2D batch, got {tuple(x.shape)}"
            )

        z = self.encoder(x)
        reconstruction = self.decoder(z)
        logits = self.classifier(z).squeeze(-1)

        return reconstruction, logits, z


model = WaferAutoencoder(X_train.shape[1])

sample = torch.tensor(X_train[:8], dtype=torch.float32)
reconstructed, logits, latent = model(sample)

print("Input shape:", tuple(sample.shape))
print("Reconstruction shape:", tuple(reconstructed.shape))
print("Logits shape:", tuple(logits.shape))
print("Latent shape:", tuple(latent.shape))

assert reconstructed.shape == sample.shape
assert logits.shape == (len(sample),)
assert latent.shape[0] == len(sample)


SECTION 4: BUILD MODEL
Input shape: (8, 471)
Reconstruction shape: (8, 471)
Logits shape: (8,)
Latent shape: (8, 32)


In [6]:
print("\nSECTION 5: TRAIN MODEL")

train_data = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

positive_count = max(int(y_train.sum()), 1)
negative_count = len(y_train) - int(y_train.sum())

positive_weight = torch.tensor(
    [negative_count / positive_count],
    dtype=torch.float32
)

bce = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
mse = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in loader:
        optimizer.zero_grad()

        reconstructed, batch_logits, _ = model(batch_x)

        assert reconstructed.shape == batch_x.shape
        assert batch_logits.shape == batch_y.shape

        reconstruction_loss = mse(reconstructed, batch_x)
        classification_loss = bce(batch_logits, batch_y)
        loss = reconstruction_loss + classification_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_x)

    print(
        f"Epoch {epoch + 1}/{epochs}: "
        f"{total_loss / len(train_data):.4f}"
    )


SECTION 5: TRAIN MODEL
Epoch 1/20: 2.2752
Epoch 2/20: 2.1253
Epoch 3/20: 1.7795
Epoch 4/20: 1.3550
Epoch 5/20: 1.1130
Epoch 6/20: 1.0124
Epoch 7/20: 1.0570
Epoch 8/20: 0.9556
Epoch 9/20: 0.9231
Epoch 10/20: 0.9141
Epoch 11/20: 0.9040
Epoch 12/20: 0.8853
Epoch 13/20: 1.0406
Epoch 14/20: 0.9574
Epoch 15/20: 0.8666
Epoch 16/20: 0.8321
Epoch 17/20: 0.8222
Epoch 18/20: 0.8148
Epoch 19/20: 0.8062
Epoch 20/20: 0.7980


In [11]:
print("\nSECTION 6: SELECT THRESHOLD ON VALIDATION DATA")

model.eval()

with torch.no_grad():
    val_tensor = torch.tensor(X_val, dtype=torch.float32)
    val_logits = model(val_tensor)[1]
    val_prob = torch.sigmoid(val_logits).numpy()

thresholds = np.arange(0.01, 0.96, 0.01)
threshold_results = []

for candidate in thresholds:
    val_pred = (val_prob >= candidate).astype(int)
    score = f1_score(y_val, val_pred, zero_division=0)
    threshold_results.append((score, candidate))

best_f1, best_threshold = max(
    threshold_results,
    key=lambda result: (result[0], -result[1])
)

threshold = float(best_threshold)

print("Selected threshold:", threshold)
print("Validation F1:", float(best_f1))


SECTION 6: SELECT THRESHOLD ON VALIDATION DATA
Selected threshold: 0.01
Validation F1: 0.0


In [8]:
print("\nSECTION 7: EVALUATE TEST DATA")

with torch.no_grad():
    test_tensor = torch.tensor(X_test, dtype=torch.float32)
    test_logits = model(test_tensor)[1]
    test_prob = torch.sigmoid(test_logits).numpy()

test_pred = (test_prob >= threshold).astype(int)

print("Test records:", len(y_test))
print("Test defects:", int(y_test.sum()))
print("Test natural defect prevalence:", float(y_test.mean()))
print("Accuracy:", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred, zero_division=0))
print("Recall:", recall_score(y_test, test_pred, zero_division=0))
print("F1:", f1_score(y_test, test_pred, zero_division=0))
print("Average precision:", average_precision_score(y_test, test_prob))

if len(np.unique(y_test)) == 2:
    print("ROC-AUC:", roc_auc_score(y_test, test_prob))
else:
    print("ROC-AUC: unavailable because test labels contain one class")

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))


SECTION 7: EVALUATE TEST DATA
Test records: 314
Test defects: 17
Test natural defect prevalence: 0.054140127388535034
Accuracy: 0.9331210191082803
Precision: 0.0
Recall: 0.0
F1: 0.0
Average precision: 0.09477163984855201
ROC-AUC: 0.6777579718756189
Confusion matrix:
[[293   4]
 [ 17   0]]


In [9]:
print("\nSECTION 8: SAVE MODEL AND PREPROCESSOR")

ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_dim": X_train.shape[1],
        "latent_dim": 32,
        "threshold": threshold,
        "usable_cols": usable_cols,
        "flag_cols": flag_cols
    },
    ARTIFACTS_PATH / "wafer_model.pt"
)

joblib.dump(
    {
        "imputer": imputer,
        "scaler": scaler,
        "usable_cols": usable_cols,
        "flag_cols": flag_cols
    },
    ARTIFACTS_PATH / "preprocessor.joblib"
)

print("Model saved:", ARTIFACTS_PATH / "wafer_model.pt")
print("Preprocessor saved:", ARTIFACTS_PATH / "preprocessor.joblib")


SECTION 8: SAVE MODEL AND PREPROCESSOR
Model saved: a:\05_Wafer\wafer-guard\artifacts\wafer_model.pt
Preprocessor saved: a:\05_Wafer\wafer-guard\artifacts\preprocessor.joblib


In [10]:
print("\nSECTION 9: AUDIT SUMMARY")

print("Evaluation split: chronological 70% / 10% / 20%")
print("Test set balanced artificially: No")
print("Preprocessing fitted on training data only: Yes")
print("Legacy sensor_1 codes used as continuous values: No")
print("Model deployment recommendation: Do not deploy without acceptable defect recall.")


model.eval()

with torch.no_grad():
    test_tensor = torch.tensor(X_test, dtype=torch.float32)
    test_logits = model(test_tensor)[1]
    test_prob = torch.sigmoid(test_logits).numpy()

test_pred = (test_prob >= threshold).astype(int)

print("TEST EVALUATION")
print("Test records:", len(y_test))
print("Actual defects:", int(y_test.sum()))
print("Test prevalence:", round(float(y_test.mean()), 4))
print("Threshold:", float(threshold))
print("Accuracy:", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred, zero_division=0))
print("Recall:", recall_score(y_test, test_pred, zero_division=0))
print("F1:", f1_score(y_test, test_pred, zero_division=0))
print("Average precision:", average_precision_score(y_test, test_prob))
print("ROC-AUC:", roc_auc_score(y_test, test_prob))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))


SECTION 9: AUDIT SUMMARY
Evaluation split: chronological 70% / 10% / 20%
Test set balanced artificially: No
Preprocessing fitted on training data only: Yes
Legacy sensor_1 codes used as continuous values: No
Model deployment recommendation: Do not deploy without acceptable defect recall.
TEST EVALUATION
Test records: 314
Actual defects: 17
Test prevalence: 0.0541
Threshold: 0.01
Accuracy: 0.9331210191082803
Precision: 0.0
Recall: 0.0
F1: 0.0
Average precision: 0.09477163984855201
ROC-AUC: 0.6777579718756189
Confusion matrix:
[[293   4]
 [ 17   0]]
